# Módulo 2 — Entrada Atmosférica (Entry)
**Programa:** Inspira STEM 2026  
**Instructores:** Oscar Tejada y Patricia Ortíz

---

## Dinámica del Frenado Aerodinámico
El 6 de agosto de 2012, la cápsula de la misión Mars Science Laboratory (Curiosity) impactó las capas superiores de la atmósfera marciana a una velocidad de 5.8 km/s (Mach 24). En esta fase, el vehículo utiliza la atmósfera planetaria como un sistema de frenado termodinámico, disipando el 99% de su energía cinética mediante fricción extrema antes de desplegar el paracaídas.

El objetivo de este módulo es modelar computacionalmente cómo las decisiones de diseño aerodinámico del vehículo interactúan con las propiedades físicas del planeta.

En los gráficos interactivos, el **diamante amarillo ($\diamond$)** representa los parámetros telemétricos reales alcanzados por el Curiosity en Marte, sirviendo como hito de calibración para sus diseños.

---

### Paso 1: Configuración de la Computadora de Vuelo
Ejecuta la siguiente celda para inicializar el motor de integración numérica y cargar las variables planetarias. Asegúrate de registrar los datos del entorno atmosférico asignado a tu equipo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact

plt.rcParams.update({'figure.dpi': 100, 'font.size': 10, 'axes.grid': True, 'grid.alpha': 0.3, 'lines.linewidth': 2})

# ==========================================
# ⚙️ DATOS DEL PLANETA ASIGNADO
# ==========================================
g_planeta = 3.711       # Gravedad local [m/s^2]
rho_planeta = 0.020     # Densidad superficial [kg/m^3]
h_scale_planeta = 11100 # Escala atmosférica [m]

def simular_entry(m, cd, area, gamma_deg):
    gamma = np.radians(gamma_deg)
    dt, h, v, t = 0.1, 125000.0, 5845.0, 0.0
    tel = {"t": [], "h": [], "v": [], "rho": [], "q": [], "drag": [], "g_load": []}
    while h >= 0.0 and t < 500:
        rho = rho_planeta * np.exp(-max(h, 0.0) / h_scale_planeta) if rho_planeta > 0 else 0.0
        q = 0.5 * rho * v**2
        drag = q * (cd * area)
        for k, val in zip(["t","h","v","rho","q","drag","g_load"], [t, h, v, rho, q, drag, drag/(m*9.81)]):
            tel[k].append(val)
        v += (g_planeta * np.sin(gamma) - (drag / m)) * dt
        h += -v * np.sin(gamma) * dt
        t += dt
        if h <= 10000.0 and gamma_deg != 90.0: break
    return {k: np.array(v) for k, v in tel.items()}

print("Motor de integración matemática inicializado y operativo.")

---
### Análisis 1: Fuerza de Arrastre vs. Área Frontal
La capacidad de la aeronave para extraer energía cinética del sistema recae en la Fuerza de Arrastre ($D$). En la misión MSL, la NASA diseñó un escudo térmico de 4.5 metros de diámetro, el más grande jamás enviado al espacio interplanetario en su momento, para maximizar este valor.

Esta fuerza es directamente proporcional a la geometría del escudo frontal ($\mathbf{A}$):
$$D = \frac{1}{2}\rho(h)v^2 C_d \mathbf{A}$$

Mediante el entorno interactivo, es posible evaluar cómo la alteración del Área Frontal ($\mathbf{A}$) deforma la trayectoria aerodinámica. El modelado exitoso requiere que la curva cinemática cruce la franja verde de altitud y velocidad, garantizando las condiciones de entrega para la siguiente fase.

**Qué esperar al mover el deslizador.** El área frontal es el único término de esa ecuación que el ingeniero elige libremente: la densidad la pone el planeta y la velocidad la pone la trayectoria de llegada.

| Si el área sube | Si el área baja |
|---|---|
| Más arrastre por kilo de vehículo | Menos arrastre por kilo |
| Frena antes y más arriba | Penetra más profundo antes de frenar |
| La curva se desplaza hacia la izquierda y termina más lenta | La curva se estira y termina más rápida |
| Menos presión sobre el escudo | Más presión sobre el escudo |

Observa dónde deja de compensar: a partir de cierta área, duplicarla ya casi no mueve el punto final de la curva. Pregúntate por qué, y qué le cuesta a la nave cada metro cuadrado extra.

In [ ]:
def interact_area(area_m2):
    d = simular_entry(m=3257.0, cd=1.45, area=area_m2, gamma_deg=14.5)
    fig, axs = plt.subplots(1, 2, figsize=(14, 4.5), constrained_layout=True)
    
    axs[0].plot(d["rho"], d["drag"]/1000, color="#2980b9")
    axs[0].set(title="Fuerza de Frenado vs Densidad Local", xlabel="Densidad $\\rho$ [kg/m³]", ylabel="Drag [kN]")
    
    axs[1].plot(d["v"]/1000, d["h"]/1000, color="#2980b9")
    axs[1].axvspan(0.35, 0.65, color="#27ae60", alpha=0.15, label="Ventana de Despliegue")
    axs[1].scatter(0.40, 10.0, color="gold", marker="D", s=100, edgecolor="k", zorder=5, label="Referencia Curiosity")
    axs[1].set(title="Trayectoria de Descenso", xlabel="Velocidad [km/s]", ylabel="Altitud [km]")
    axs[1].invert_yaxis(); axs[1].legend()
    plt.show()

interact(interact_area, area_m2=widgets.FloatSlider(value=15.9, min=2.0, max=40.0, step=0.5, description='Área [m²]:'));

---
### Análisis 2: Ángulo de Vuelo y Estrés Inercial
El vehículo requiere un corredor de aproximación altamente preciso. Curiosity ingresó con un Ángulo de Trayectoria ($\mathbf{\gamma}$) superficial de aproximadamente 14.5 grados.

Un ángulo de vuelo pronunciado incrementa dramáticamente la derivada de la altitud, obligando al vehículo a frenar en las capas densas de la atmósfera en un intervalo de tiempo muy corto. Esto induce picos de estrés inercial ($n_z$) que pueden fracturar la estructura térmica y los instrumentos científicos:
$$ \frac{dh}{dt} = -v \sin(\mathbf{\gamma}) $$
$$n_z = \frac{D}{m \cdot g_0}$$

Modificando el Ángulo de entrada ($\mathbf{\gamma}$), se puede visualizar computacionalmente cómo un perfil de vuelo aerodinámico superficial distribuye la carga inercial de la desaceleración, manteniéndola por debajo de las tolerancias de hardware.

> **Nota sobre el modelo.** Con el ángulo real de 14,5° la curva del Análisis 1 termina lejos del diamante dorado. No es un error de los datos: este modelo cae en línea recta con ángulo constante, mientras que MSL voló una entrada guiada con sustentación, inclinando la cápsula y alternando el banqueo para estirar la trayectoria. Ese alargamiento equivale, en un modelo sin sustentación, a entrar mucho más plano: baja el deslizador a 6,7° y la curva pasa por el punto real. La diferencia entre 14,5° y 6,7° es lo que aporta el control de la entrada.

**Qué esperar al mover el deslizador.** El ángulo no cambia cuánta energía hay que disipar, solo en cuánto espacio y en cuánto tiempo se disipa.

| Si el ángulo sube | Si el ángulo baja |
|---|---|
| El vehículo cruza la atmósfera en menos recorrido | El recorrido se alarga |
| Todo el frenado se concentra en las capas densas | El frenado se reparte en capas tenues |
| El pico de carga sube casi en proporción al seno del ángulo | El pico de carga baja |
| Llega más abajo y más rápido | Llega más arriba y más lento |

Prueba 5°, 15° y 45° y anota el pico de cada uno. La relación entre ellos no es casual.

In [ ]:
def interact_angulo(angulo_deg):
    d = simular_entry(m=3257.0, cd=1.45, area=15.9, gamma_deg=angulo_deg)
    fig, axs = plt.subplots(1, 2, figsize=(14, 4.5), constrained_layout=True)
    
    axs[0].plot(d["h"]/1000, d["g_load"], color="#8e44ad")
    axs[0].set(title="Estrés Inercial por Altitud", xlabel="Altitud [km]", ylabel="Fuerzas G [G]")
    axs[0].invert_xaxis()
    
    axs[1].plot(d["t"], d["g_load"], color="#8e44ad")
    axs[1].axhline(15.0, color="k", linestyle="--", label="Límite Estructural (15 G)")
    axs[1].set(title="Evolución Transitoria de Cargas", xlabel="Tiempo [s]")
    axs[1].legend()
    plt.show()

interact(interact_angulo, angulo_deg=widgets.FloatSlider(value=14.5, min=4.0, max=90.0, step=0.1, description='Ángulo [°]:'));

---
### Análisis 3: Inercia y Presión Dinámica Térmica
Curiosity fue el rover más pesado jamás enviado a Marte (con una masa inicial de entrada superior a las 3.2 toneladas). A velocidades hipersónicas, la fricción atmosférica genera un escudo de plasma y una Presión Dinámica ($q$) masiva.

El tiempo de exposición a este régimen de calor ablativo depende de la inercia del vehículo, cuantificada en el diseño aeroespacial a través del Coeficiente Balístico ($\beta$):
$$\beta = \frac{\mathbf{m}}{C_dA} \quad [\text{kg/m}^2]$$

Al manipular paramétricamente la Masa ($\mathbf{m}$) del vehículo, se evalúa cómo el incremento inercial obliga a la cápsula a penetrar más profundo en la atmósfera antes de perder velocidad, exponiendo el escudo térmico (TPS) a presiones dinámicas que pueden superar los límites del material.

**Qué esperar al mover el deslizador.** La masa aparece en el numerador del coeficiente balístico, así que hace exactamente lo contrario del área.

| Si la masa sube | Si la masa baja |
|---|---|
| El coeficiente balístico sube | El coeficiente balístico baja |
| La cápsula penetra más antes de frenar | Frena antes, en aire más tenue |
| La presión sobre el escudo sube | La presión baja |
| El pico de carga en g… obsérvalo | …y compáralo con el anterior |

Esa última fila es la parte interesante. Mueve la masa de 1 000 a 7 000 kg y vigila el pico de g de la gráfica derecha del Análisis 2: cambia mucho menos de lo que cabría esperar. Compáralo con lo que le pasa a la presión dinámica en esta misma pantalla. Dos indicadores del mismo vehículo, y solo uno responde al rediseño.

In [ ]:
def interact_masa(masa_kg):
    d = simular_entry(m=masa_kg, cd=1.45, area=15.9, gamma_deg=14.5)
    fig, axs = plt.subplots(1, 2, figsize=(14, 4.5), constrained_layout=True)
    
    axs[0].plot(d["v"]/1000, d["q"]/1000, color="#c0392b")
    axs[0].set(title="Presión Térmica frente a Velocidad", xlabel="Velocidad [km/s]", ylabel="Presión Dinámica $q$ [kPa]")
    axs[0].invert_xaxis()
    
    axs[1].plot(d["t"], d["q"]/1000, color="#c0392b")
    axs[1].axhline(25.0, color="k", linestyle="--", label="Tolerancia del Escudo Térmico (25 kPa)")
    axs[1].set(title="Exposición Térmica en el Tiempo", xlabel="Tiempo [s]")
    axs[1].legend()
    plt.show()

interact(interact_masa, masa_kg=widgets.FloatSlider(value=3257.0, min=1000.0, max=7000.0, step=100.0, description='Masa [kg]:'));

### Análisis de Grupo
Reúnanse y debatan los siguientes escenarios de ingeniería orbital, apoyándose en la matemática del simulador:

1. **El Límite del Escudo Térmico (Misión Real):** En Marte, el escudo térmico de Curiosity (hecho de un material llamado PICA-X) soportó temperaturas de casi 2,100 °C. Si la atmósfera de su planeta asignado fuera mucho más tenue que la marciana, ¿qué ajuste estructural están obligados a realizar en el Área Frontal (Análisis 1) para lograr detenerse, y cómo afectaría físicamente ese rediseño a la construcción de la nave?
2. **El Corredor de Entrada Segura:** Un error de navegación que resulte en un ángulo de entrada demasiado superficial hará que la nave "rebote" en la atmósfera (como una piedra sobre el agua). Por el contrario, observando la derivada de altitud en el Análisis 2, expliquen el mecanismo físico que ocasiona que un ángulo de aproximación muy pronunciado (ej. 90 grados) fracture la nave antes de tocar el suelo, a pesar de usar exactamente el mismo escudo térmico.
3. **La Penalización de la Carga Útil:** La tendencia en la exploración espacial es enviar laboratorios móviles cada vez más pesados y complejos. Mirando la relación de la masa en la ecuación del Coeficiente Balístico (Análisis 3), si la masa de las futuras misiones se duplica y corremos el riesgo de derretir la nave, ¿de qué manera los ingenieros aeronáuticos deben alterar las variables del Análisis 1 para compensar este incremento inercial?